In [23]:
import pandas as pd
import numpy as np

In [24]:
from google.cloud import bigquery

PROJECT = "gridzero-489711"
DATASET = "merged_set"
TABLE = "test_merge_2017_onward_raw"

query = f"""
    SELECT *
    FROM {PROJECT}.{DATASET}.{TABLE}
"""

client = bigquery.Client('gridzero-489711')
query_job = client.query(query)
result = query_job.result()
df = result.to_dataframe()

/Users/jamesla/.pyenv/versions/3.12.9/envs/gridzero/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [25]:
df.head()

,datetime,temperature_2m_c,wind_speed_100m_ms,wind_gusts_10m_ms,cloud_cover_pct,shortwave_radiation_wm2,direct_radiation_wm2,diffuse_radiation_wm2,pressure_msl_hpa,snowfall_cm,...,Hydro Pumped Storage,Hydro Run-of-river and poundage,Nuclear,Other,Solar,Wind Offshore,Wind Onshore,TotalOutput-MW,carbon_intensity_gCO2_kWh,status
0,2017-09-12 00:00:00,11.6,31.0,28.1,4,0.0,0.0,0.0,1001.2,0.0,...,0.0,265.0,7897.0,173.0,0.0,4006.573,4394.973,21722.546,142.0,okay
1,2017-09-12 00:30:00,11.6,31.0,28.1,4,0.0,0.0,0.0,1001.2,0.0,...,0.0,248.0,7897.0,174.0,0.0,3973.985,4418.446,21554.431,140.0,okay
2,2017-09-12 01:00:00,11.2,30.3,27.0,5,0.0,0.0,0.0,1001.9,0.0,...,0.0,242.0,7852.0,171.0,0.0,3941.698,4533.019,21767.717,139.0,okay
3,2017-09-12 01:30:00,11.2,30.3,27.0,5,0.0,0.0,0.0,1001.9,0.0,...,0.0,242.0,7706.0,174.0,0.0,3945.620,4726.191,21742.811,137.0,okay
4,2017-09-12 02:00:00,10.9,29.6,25.2,7,0.0,0.0,0.0,1002.4,0.0,...,0.0,242.0,7684.0,173.0,0.0,3919.454,4832.410,21579.864,132.0,okay


In [26]:
df.columns

Index(['datetime', 'temperature_2m_c', 'wind_speed_100m_ms',
       'wind_gusts_10m_ms', 'cloud_cover_pct', 'shortwave_radiation_wm2',
       'direct_radiation_wm2', 'diffuse_radiation_wm2', 'pressure_msl_hpa',
       'snowfall_cm', 'rain_mm', 'precipitation_mm', 'Biomass', 'Fossil Gas',
       'Fossil Hard coal', 'Fossil Oil', 'Hydro Pumped Storage',
       'Hydro Run-of-river and poundage', 'Nuclear', 'Other', 'Solar',
       'Wind Offshore', 'Wind Onshore', 'TotalOutput-MW',
       'carbon_intensity_gCO2_kWh', 'status'],
      dtype='object')

# Cleaning Data

In [27]:
df['datetime'].diff().value_counts()

datetime
0 days 00:30:00    148990
Name: count, dtype: int64

In [28]:
weather_cols = [
'temperature_2m_c','wind_speed_100m_ms','wind_gusts_10m_ms',
'cloud_cover_pct','shortwave_radiation_wm2','direct_radiation_wm2',
'diffuse_radiation_wm2','pressure_msl_hpa'
]

df[weather_cols] = df[weather_cols].interpolate(method='linear')

In [29]:
gen_cols = [
'Biomass','Fossil Gas','Fossil Hard coal','Fossil Oil',
'Hydro Pumped Storage','Hydro Run-of-river and poundage',
'Nuclear','Other','Solar','Wind Offshore','Wind Onshore'
]

df[gen_cols] = df[gen_cols].interpolate()

df['TotalOutput-MW'] = df[gen_cols].sum(axis=1)

In [30]:
# we have precipitation already
df = df.drop(columns=['rain_mm','snowfall_cm', 'Fossil Oil'])

In [31]:
# Create Time Features
df['hour'] = df['datetime'].dt.hour
df['day_of_week'] = df['datetime'].dt.dayofweek
df['day_of_year'] = df['datetime'].dt.dayofyear

In [32]:
# cyclical encoding
df['hour_sin'] = np.sin(2*np.pi*df['hour']/24)
df['hour_cos'] = np.cos(2*np.pi*df['hour']/24)

df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

df['doy_sin'] = np.sin(2*np.pi*df['day_of_year']/365)
df['doy_cos'] = np.cos(2*np.pi*df['day_of_year']/365)

In [39]:
df = df.drop(columns=['hour','day_of_week', 'day_of_year', 'status'])

In [33]:
lags = [48, 336, 17520]   # 30min steps

for lag in lags:
    df[f'carbon_lag_{lag}'] = df['carbon_intensity_gCO2_kWh'].shift(lag)

In [34]:
df.columns

Index(['datetime', 'temperature_2m_c', 'wind_speed_100m_ms',
       'wind_gusts_10m_ms', 'cloud_cover_pct', 'shortwave_radiation_wm2',
       'direct_radiation_wm2', 'diffuse_radiation_wm2', 'pressure_msl_hpa',
       'precipitation_mm', 'Biomass', 'Fossil Gas', 'Fossil Hard coal',
       'Hydro Pumped Storage', 'Hydro Run-of-river and poundage', 'Nuclear',
       'Other', 'Solar', 'Wind Offshore', 'Wind Onshore', 'TotalOutput-MW',
       'carbon_intensity_gCO2_kWh', 'status', 'hour', 'day_of_week',
       'day_of_year', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'doy_sin',
       'doy_cos', 'carbon_lag_48', 'carbon_lag_336', 'carbon_lag_17520'],
      dtype='object')

In [35]:
df.columns = (
    df.columns
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('-', '_')
)

In [36]:
df.columns

Index(['datetime', 'temperature_2m_c', 'wind_speed_100m_ms',
       'wind_gusts_10m_ms', 'cloud_cover_pct', 'shortwave_radiation_wm2',
       'direct_radiation_wm2', 'diffuse_radiation_wm2', 'pressure_msl_hpa',
       'precipitation_mm', 'biomass', 'fossil_gas', 'fossil_hard_coal',
       'hydro_pumped_storage', 'hydro_run_of_river_and_poundage', 'nuclear',
       'other', 'solar', 'wind_offshore', 'wind_onshore', 'totaloutput_mw',
       'carbon_intensity_gco2_kwh', 'status', 'hour', 'day_of_week',
       'day_of_year', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'doy_sin',
       'doy_cos', 'carbon_lag_48', 'carbon_lag_336', 'carbon_lag_17520'],
      dtype='object')

In [37]:
df.isnull().sum()

datetime                               0
temperature_2m_c                       0
wind_speed_100m_ms                     0
wind_gusts_10m_ms                      0
cloud_cover_pct                        0
shortwave_radiation_wm2                0
direct_radiation_wm2                   0
diffuse_radiation_wm2                  0
pressure_msl_hpa                       0
precipitation_mm                       0
biomass                                0
fossil_gas                             0
fossil_hard_coal                       0
hydro_pumped_storage                   0
hydro_run_of_river_and_poundage        0
nuclear                                0
other                                  0
solar                                  0
wind_offshore                          0
wind_onshore                           0
totaloutput_mw                         0
carbon_intensity_gco2_kwh            226
status                                 0
hour                                   0
day_of_week     

In [40]:
# from sklearn.impute import KNNImputer

# imputer = KNNImputer(n_neighbors=5)

# df_imputed = pd.DataFrame(
#     imputer.fit_transform(df.drop(columns=['datetime'])),
#     columns=df.drop(columns=['datetime']).columns
# )

# df_imputed['datetime'] = df['datetime']

In [57]:
# Recalculate Carbon Intensity From Generation Mix using approximate typical emission factors

emissions = (
    df['biomass'] * 230 +
    df['fossil_gas'] * 490 +
    df['fossil_hard_coal'] * 820 +
    df['nuclear'] * 12 +
    df['solar'] * 45 +
    df['wind_onshore'] * 11 +
    df['wind_offshore'] * 11 +
    df['hydro_run_of_river_and_poundage'] * 24
)

df['carbon_estimate'] = emissions / df['totaloutput_mw']
df['carbon_intensity_gco2_kwh'] = df['carbon_intensity_gco2_kwh'].fillna(df['carbon_estimate'])
df = df.drop(columns='carbon_estimate')

In [68]:
df.columns

Index(['datetime', 'temperature_2m_c', 'wind_speed_100m_ms',
       'wind_gusts_10m_ms', 'cloud_cover_pct', 'shortwave_radiation_wm2',
       'direct_radiation_wm2', 'diffuse_radiation_wm2', 'pressure_msl_hpa',
       'precipitation_mm', 'biomass', 'fossil_gas', 'fossil_hard_coal',
       'hydro_pumped_storage', 'hydro_run_of_river_and_poundage', 'nuclear',
       'other', 'solar', 'wind_offshore', 'wind_onshore', 'totaloutput_mw',
       'carbon_intensity_gco2_kwh', 'hour_sin', 'hour_cos', 'dow_sin',
       'dow_cos', 'doy_sin', 'doy_cos', 'carbon_lag_48', 'carbon_lag_336',
       'carbon_lag_17520'],
      dtype='object')